# Karpathy's Autoresearch on Google Colab
**Autonomous AI research agent** — trains a GPT model and lets an AI iterate on `train.py` to improve it.

**Setup:** Runtime → Change runtime type → **GPU** (T4 free, or A100 with Pro)

In [ ]:
# Step 1: Check GPU
!nvidia-smi
import torch
print(f"\nGPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"Compute capability: {torch.cuda.get_device_capability()}")

In [ ]:
# Step 2: Clone repo and install deps
!git clone https://github.com/karpathy/autoresearch.git /content/autoresearch
!pip install -q kernels rustbpe tiktoken pyarrow

In [ ]:
# Step 3: Download data and train tokenizer
!python /content/autoresearch/prepare.py --num-shards 2

In [ ]:
# Step 4: Patch train.py for non-H100 GPUs
import torch, math, re, os

cap = torch.cuda.get_device_capability()
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
has_bf16 = cap >= (8, 0)

print(f"GPU: sm_{cap[0]}{cap[1]}, VRAM: {vram_gb:.1f}GB, bf16: {has_bf16}")

if vram_gb >= 70:   batch_size, depth = 128, 8
elif vram_gb >= 35: batch_size, depth = 64, 8
elif vram_gb >= 20: batch_size, depth = 32, 8
elif vram_gb >= 14: batch_size, depth = 16, 8
else:               batch_size, depth = 4, 4

total_batch = 2 ** int(math.ceil(math.log2(max(batch_size * 2048, 2**15))))
print(f"Settings: batch={batch_size}, depth={depth}, total_batch={total_batch}")

train_py = '/content/autoresearch/train.py'
with open(train_py) as f: code = f.read()

# Replace FA3 with SDPA
code = code.replace(
    'from kernels import get_kernel\n'
    'cap = torch.cuda.get_device_capability()\n'
    '# varunneal\'s FA3 is Hopper only, use kernels-community on non-Hopper GPUs\n'
    'repo = "varunneal/flash-attention-3" if cap == (9, 0) else "kernels-community/flash-attn3"\n'
    'fa3 = get_kernel(repo).flash_attn_interface',
    '# Using PyTorch SDPA instead of FA3 for broad GPU compat')

code = code.replace(
    '        y = fa3.flash_attn_func(q, k, v, causal=True, window_size=window_size)\n'
    '        y = y.contiguous().view(B, T, -1)',
    '        q = q.transpose(1, 2)\n'
    '        k = k.transpose(1, 2)\n'
    '        v = v.transpose(1, 2)\n'
    '        if self.n_kv_head < self.n_head:\n'
    '            reps = self.n_head // self.n_kv_head\n'
    '            k = k.repeat_interleave(reps, dim=1)\n'
    '            v = v.repeat_interleave(reps, dim=1)\n'
    '        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)\n'
    '        y = y.transpose(1, 2).contiguous().view(B, T, -1)')

# fp16 fallback for non-Ampere GPUs
if not has_bf16:
    code = code.replace('torch.bfloat16', 'torch.float16')
    code = code.replace(
        'model = torch.compile(model, dynamic=False)',
        'scaler = torch.amp.GradScaler()\nmodel = torch.compile(model, dynamic=False)')
    code = code.replace(
        '        loss = loss / grad_accum_steps\n        loss.backward()',
        '        loss = loss / grad_accum_steps\n        scaler.scale(loss).backward()')
    code = code.replace(
        '    optimizer.step()\n    model.zero_grad(set_to_none=True)',
        '    scaler.unscale_(optimizer)\n    scaler.step(optimizer)\n    scaler.update()\n    model.zero_grad(set_to_none=True)')

# Set sizes
code = re.sub(r'DEVICE_BATCH_SIZE = \d+.*', f'DEVICE_BATCH_SIZE = {batch_size}', code)
code = re.sub(r'DEPTH = \d+.*', f'DEPTH = {depth}', code)
code = re.sub(r'TOTAL_BATCH_SIZE = [^\n]+', f'TOTAL_BATCH_SIZE = {total_batch}', code)

with open(train_py, 'w') as f: f.write(code)
print('train.py patched!')

In [ ]:
# Step 5: Run baseline training (~5 min)
!python /content/autoresearch/train.py

## Next steps
Once baseline runs, you can iterate:
1. **Edit train.py** in the Colab file browser (left panel)
2. **Re-run Step 5** to test your changes
3. **Lower val_bpb = better** — that's the only metric that matters